In [376]:
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

In [377]:
path = r"data\dataset_elo.csv"
with open(path) as file:
    df = pd.read_csv(file)
df.head()

,year,month,day,home_team,away_team,stage,result,shootouts,home_score,home_is_host,...,away_total_market_value_eur,away_avg_age,away_wc_participations_before,away_groups_passed_before,away_round16_before,away_quarterfinals_before,away_semifinals_before,away_finals_before,elo_home,elo_away
0,2006,6,9,Germany,Costa Rica,0,0.0,0,4.0,1,...,18400000.0,24.7,2,1,1,0,0,0,1500.0,1500.0
1,2006,6,9,Poland,Ecuador,0,1.0,0,0.0,0,...,387830000.0,25.7,1,0,0,0,0,0,1500.0,1500.0
2,2006,6,10,Argentina,Ivory Coast,0,0.0,0,2.0,0,...,410900000.0,25.8,0,0,0,0,0,0,1500.0,1500.0
3,2006,6,10,England,Paraguay,0,0.0,0,1.0,0,...,141850000.0,29.1,6,3,3,0,0,0,1500.0,1500.0
4,2006,6,10,Trinidad and Tobago,Sweden,0,2.0,0,0.0,0,...,363780000.0,26.3,10,7,3,4,3,1,1500.0,1500.0


In [378]:
# selecting features and target
features = df[['year', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y', 'home_wc_titles_before',
       'home_total_market_value_eur', 'home_avg_age',
       'home_wc_participations_before', 'home_groups_passed_before',
       'home_round16_before', 'home_quarterfinals_before',
       'home_semifinals_before', 'home_finals_before',
       'away_is_host', 'away_goals_scored_last_4y',
       'away_goals_received_last_4y', 'away_wins_last_4y',
       'away_losses_last_4y', 'away_draws_last_4y', 'away_wc_titles_before',
       'away_total_market_value_eur', 'away_avg_age',
       'away_wc_participations_before', 'away_groups_passed_before',
       'away_round16_before', 'away_quarterfinals_before',
       'away_semifinals_before', 'away_finals_before', 'stage', 'elo_home', 'elo_away']]
target = df[['year', 'result', 'stage']]

#### Using logistic regression

In [379]:
# standardizing features
features.columns

Index(['year', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y', 'home_wc_titles_before',
       'home_total_market_value_eur', 'home_avg_age',
       'home_wc_participations_before', 'home_groups_passed_before',
       'home_round16_before', 'home_quarterfinals_before',
       'home_semifinals_before', 'home_finals_before', 'away_is_host',
       'away_goals_scored_last_4y', 'away_goals_received_last_4y',
       'away_wins_last_4y', 'away_losses_last_4y', 'away_draws_last_4y',
       'away_wc_titles_before', 'away_total_market_value_eur', 'away_avg_age',
       'away_wc_participations_before', 'away_groups_passed_before',
       'away_round16_before', 'away_quarterfinals_before',
       'away_semifinals_before', 'away_finals_before', 'stage', 'elo_home',
       'elo_away'],
      dtype='str')

In [380]:
scaler = StandardScaler()
to_be_standardized = ["home_goals_scored_last_4y", "home_goals_received_last_4y", "home_wins_last_4y", "home_losses_last_4y",
                        "home_draws_last_4y", "home_wc_titles_before", "home_total_market_value_eur", "home_avg_age", 
                        "home_wc_participations_before", "home_groups_passed_before", "home_round16_before", "home_quarterfinals_before",
                        "home_semifinals_before", "home_finals_before", 'away_goals_scored_last_4y', 'away_goals_received_last_4y',
                        'away_wins_last_4y', 'away_losses_last_4y', 'away_draws_last_4y',
                        'away_wc_titles_before', 'away_total_market_value_eur', 'away_avg_age',
                        'away_wc_participations_before', 'away_groups_passed_before',
                        'away_round16_before', 'away_quarterfinals_before',
                        'away_semifinals_before', 'away_finals_before', 'elo_home', 'elo_away']


In [381]:
# Accuracy and loss are computed on the train set

# to be modified: now it outputs a binary variable (it is totally wrong), I should implement a softmax regression with 3 classes
# try a ridge regularization and see what happens

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target.loc[target['year'] != 2026, 'result']

X_test = features[(features['year'] == 2026)].drop(columns=['year'])
Y_test = target.loc[target['year'] == 2026, 'result']

# standardizing features
X_train_scaled = X_train.copy()
X_train_scaled[to_be_standardized] = X_train_scaled[to_be_standardized].astype(float)
X_train_scaled.loc[:, to_be_standardized] = scaler.fit_transform(X_train_scaled[to_be_standardized])

X_test_scaled = X_test.copy()
X_test_scaled[to_be_standardized] = X_test_scaled[to_be_standardized].astype(float)
X_test_scaled.loc[:, to_be_standardized] = scaler.transform(X_test_scaled[to_be_standardized])

model = LogisticRegression(max_iter=100)
_ = model.fit(X_train_scaled, Y_train)

prob = _.predict_proba(X_train_scaled)
preds = _.predict(X_train_scaled)

acc = accuracy_score(Y_train, preds)
loss = log_loss(Y_train, prob)

print(f"World Cup 2006-2022: Accuracy: {acc:.4f}, LogLoss: {loss:.4f}")

World Cup 2006-2022: Accuracy: 0.6438, LogLoss: 0.7832


In [382]:
# computing accuracy and loss on test set

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target[(target['year'] != 2026)]['result']

X_test = features[(features['year'] == 2026)].drop(columns=['year'])
Y_test = target[(target['year'] == 2026)]['result']

# standardizing features
X_train_scaled = X_train.copy()
X_train_scaled[to_be_standardized] = X_train_scaled[to_be_standardized].astype(float)
X_train_scaled.loc[:, to_be_standardized] = scaler.fit_transform(X_train_scaled[to_be_standardized])

X_test_scaled = X_test.copy()
X_test_scaled[to_be_standardized] = X_test_scaled[to_be_standardized].astype(float)
X_test_scaled.loc[:, to_be_standardized] = scaler.transform(X_test_scaled[to_be_standardized])

model = LogisticRegression(max_iter=100)
_ = model.fit(X_train_scaled, Y_train)

prob = _.predict_proba(X_test_scaled)
preds = _.predict(X_test_scaled)

acc = accuracy_score(Y_test, preds)
loss = log_loss(Y_test, prob)

print(f"World Cup 2026: Accuracy: {acc:.4f}, LogLoss: {loss:.4f}")

World Cup 2026: Accuracy: 0.6100, LogLoss: 0.8548


#### Trying XGBoost
Here I do not standardize the features as it is totally irrelevant for XGBoost

In [383]:
# Accuracy and loss are computed on the training set

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target[(target['year'] != 2026)]['result']

X_test = features[(features['year'] == 2026)].drop(columns=['year'])
Y_test = target[(target['year'] == 2026)]['result']

# fitting model and computing probabilities
model = xgb.XGBClassifier(
        objective="multi:softprob", num_class=3,
        max_depth=3, min_child_weight=10,
        subsample=0.7, colsample_bytree=0.7,
        reg_lambda=4.0, n_estimators=10,
        eval_metric="mlogloss", random_state=42,
        n_jobs=1, tree_method="exact" # to grant reproducibility on every CPU
)
model.fit(X_train, Y_train)

prob = model.predict_proba(X_train)

# creating a dataframe to store probabilities
df_prob = pd.DataFrame(prob, columns = ["home_prob", "away_prob", "draw_prob"])
df_prob["stage"] = df[df["year"] < 2026]["stage"].values

# compute normalization only for knockouts
sum_no_draw = df_prob["home_prob"] + df_prob["away_prob"]
df_prob["home_ko_prob"] = df_prob["home_prob"] / sum_no_draw
df_prob["away_ko_prob"] = df_prob["away_prob"] / sum_no_draw

def final_output(row):
    if row["stage"] == 1:
        return [row["home_ko_prob"], row["away_ko_prob"]]
    else:
        return [row["home_prob"], row["draw_prob"], row["away_prob"]]

df_prob["probabilities"] = df_prob.apply(final_output, axis=1)

# inferring prediction from probabilities
def predict(row):
    if row["stage"] == 1:
        if row["home_ko_prob"] > row["away_ko_prob"]:
            return 0 # home_team wins
        else:
            return 1 # away_team wins
    else:
        classes = [0, 1, 2] # for groups
        groups_prob = [row["home_prob"], row["away_prob"], row["draw_prob"]]
        return classes[np.argmax(groups_prob)]

df_prob["prediction"] = df_prob.apply(predict, axis=1)
preds = df_prob["prediction"]

acc = accuracy_score(Y_train, preds)

# computing loss for groups 
is_group = df_prob["stage"] == 0
target_groups = Y_train[is_group]
probs_groups = df_prob[is_group][["home_prob", "away_prob", "draw_prob"]].values
loss_groups = log_loss(target_groups, probs_groups)

# computing loss for knockouts
is_ko = df_prob["stage"] == 1
target_ko = Y_train[is_ko]
probs_ko = df_prob[is_ko][["home_ko_prob", "away_ko_prob"]].values
loss_ko = log_loss(target_ko, probs_ko)

print(f"World Cup 2006-2022: Accuracy: {acc:.4f} | LogLoss groups: {loss_groups:.4f} | LogLoss ko: {loss_ko:.4f}")

World Cup 2006-2022: Accuracy: 0.6969 | LogLoss groups: 0.8053 | LogLoss ko: 0.4272


In [384]:
# Computing accuracy and loss on the test set

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target[(target['year'] != 2026)]['result']

X_test = features[(features['year'] == 2026)].drop(columns=['year'])
Y_test = target[(target['year'] == 2026)]['result'].reset_index(drop=True) 
# reset_index to align the index with is_group to correctly index it when computing losses
Y_test.drop(columns = "stage", inplace=True)

# fitting model and computing probabilities
model = xgb.XGBClassifier(
        objective="multi:softprob", num_class=3,
        max_depth=3, min_child_weight=10,
        subsample=0.7, colsample_bytree=0.7,
        reg_lambda=31.0, n_estimators=109,
        eval_metric="mlogloss", random_state=42,
        n_jobs=1, tree_method="exact" # to grant reproducibility on every CPU
)
model.fit(X_train, Y_train)

prob = model.predict_proba(X_test)

# creating a dataframe to store probabilities
df_prob = pd.DataFrame(prob, columns = ["home_prob", "away_prob", "draw_prob"])
df_prob["stage"] = df[df["year"] == 2026]["stage"].values

# compute normalization only for knockouts
sum_no_draw = df_prob["home_prob"] + df_prob["away_prob"]
df_prob["home_ko_prob"] = df_prob["home_prob"] / sum_no_draw
df_prob["away_ko_prob"] = df_prob["away_prob"] / sum_no_draw

def final_output(row):
    if row["stage"] == 1:
        return [row["home_ko_prob"], row["away_ko_prob"]]
    else:
        return [row["home_prob"], row["away_prob"], row["draw_prob"]]

df_prob["probabilities"] = df_prob.apply(final_output, axis=1)

# inferring prediction from probabilities
def predict(row):
    if row["stage"] == 1:
        if row["home_ko_prob"] > row["away_ko_prob"]:
            return 0 # home_team wins
        else:
            return 1 # away_team wins
    else:
        classes = [0, 1, 2] # for groups
        groups_prob = [row["home_prob"], row["away_prob"], row["draw_prob"]]
        return classes[np.argmax(groups_prob)]
    
df_prob["prediction"] = df_prob.apply(predict, axis=1)
preds = df_prob["prediction"]

acc = accuracy_score(Y_test, preds)

# computing loss for groups 
is_group = df_prob["stage"] == 0
target_groups = Y_test[is_group]
probs_groups = df_prob[is_group][["home_prob", "away_prob", "draw_prob"]].values
loss_groups = log_loss(target_groups, probs_groups)

# computing loss for knockouts
is_ko = df_prob["stage"] == 1
target_ko = Y_test[is_ko]
probs_ko = df_prob[is_ko][["home_ko_prob", "away_ko_prob"]].values
loss_ko = log_loss(target_ko, probs_ko)

print(f"World Cup 2026: Accuracy: {acc:.4f} | LogLoss groups: {loss_groups:.4f} | LogLoss ko: {loss_ko:.4f}")

World Cup 2026: Accuracy: 0.6400 | LogLoss groups: 0.9451 | LogLoss ko: 0.5496


#### Predicting only knockouts
Loss for ko matches is considerably smaller, and I want to predict 4 ko macthes. Thus I'm interested in seeing what the accuracy is if I test my model only on ko matches.

In [385]:
# Computing accuracy and loss on the test set

X_train = features[(features['year'] != 2026)].drop(columns='year')
Y_train = target[(target['year'] != 2026)]['result']

X_test = features[(features['year'] == 2026) & (features["stage"] == 1)].drop(columns='year')
Y_test = target[(target["year"] == 2026) & (target["stage"] == 1)]["result"]

# fitting model and computing probabilities
model = xgb.XGBClassifier(
        objective="multi:softprob",
        max_depth=3, min_child_weight=10,
        subsample=0.7, colsample_bytree=0.7,
        reg_lambda=31.0, n_estimators=109,
        eval_metric="mlogloss", random_state=42,
        n_jobs=1, tree_method="exact" # to grant reproducibility on every CPU
)

model.fit(X_train, Y_train)

prob = model.predict_proba(X_test)

# creating a dataframe to store probabilities
df_prob = pd.DataFrame(prob, columns = ["home_prob", "away_prob", "draw_prob"])
df_prob["stage"] = df[(df["year"] == 2026) & (df["stage"] == 1)]["stage"].values

# compute normalization for knockouts
sum_no_draw = df_prob["home_prob"] + df_prob["away_prob"]
df_prob["home_ko_prob"] = df_prob["home_prob"] / sum_no_draw
df_prob["away_ko_prob"] = df_prob["away_prob"] / sum_no_draw

def final_output(row):
    return [row["home_ko_prob"], row["away_ko_prob"]]

df_prob["probabilities"] = df_prob.apply(final_output, axis=1)

# inferring prediction from probabilities
def predict(row):
    if row["home_ko_prob"] > row["away_ko_prob"]:
        return 0 # home_team wins
    else:
        return 1 # away_team wins
    
df_prob["prediction"] = df_prob.apply(predict, axis=1)
preds = df_prob["prediction"]

acc = accuracy_score(Y_test, preds)

# computing loss for knockouts
probs_ko = df_prob[["home_ko_prob", "away_ko_prob"]].values
loss_ko = log_loss(Y_test, probs_ko)

print(f"World Cup 2026: Accuracy: {acc:.4f} | LogLoss: {loss_ko:.4f}")

World Cup 2026: Accuracy: 0.7500 | LogLoss: 0.5496


Accuracy is much better and loss did not change as I expected.

#### Optimizing hyperparameters
As I said above it is more convenient to optimize hyperparameters for only ko matches.

In [386]:
# from sklearn.model_selection import RandomizedSearchCV

# def cerca_migliore_accuracy_poi_loss(cv_results):
#     """
#     Trova il modello con l'Accuracy più alta. 
#     In caso di parità (o tolleranza dello 0.5%), sceglie quello con la Log Loss migliore.
#     """
#     # Nomi reali delle colonne generati dalla tupla di scoring
#     accuracy_medie = cv_results['mean_test_accuracy']
#     loss_medie = cv_results['mean_test_neg_log_loss']  # Numeri negativi
    
#     # Troviamo il valore massimo di accuratezza raggiunto nella gara
#     max_accuracy = np.max(accuracy_medie)
    
#     # Tolleranza dello 0.5% per includere i modelli quasi parimerito in accuratezza
#     tolleranza = 0.005 
    
#     # Prendiamo gli indici dei modelli che sono "vicini al top" dell'accuracy
#     modelli_top_idx = np.where(accuracy_medie >= (max_accuracy - tolleranza))[0]
    
#     # Tra questi, scegliamo quello con la Log Loss più alta (più vicina a zero)
#     indice_vincitore = modelli_top_idx[np.argmax(loss_medie[modelli_top_idx])]
    
#     return indice_vincitore

# xgb_base = xgb.XGBClassifier(
#     objective="multi:softprob", 
#     num_class=3,
#     eval_metric="mlogloss", 
#     random_state=42,
#     n_jobs=1, 
#     tree_method="exact"
# )

# param_distributions = {
#     'max_depth': [3, 4, 5],                       
#     'n_estimators': [80, 120, 160, 200],      
#     'min_child_weight': [5, 10, 15],              
#     'reg_lambda': [15.0, 30.0, 50.0],             
#     'subsample': [0.6, 0.7, 0.8],                 
#     'colsample_bytree': [0.6, 0.7, 0.8]           
# }

# random_search = RandomizedSearchCV(
#     estimator=xgb_base,
#     param_distributions=param_distributions,
#     n_iter=15,             
#     cv=2,                  
#     scoring=('accuracy', 'neg_log_loss'),
#     refit=cerca_migliore_accuracy_poi_loss,
#     random_state=42,
#     verbose=1              
# )

# random_search.fit(X_train, Y_train)

# print("\nTuning completato!")
# print("Migliori parametri scelti per ottimizzare la Loss:", random_search.best_params_)

# indice_migliore = random_search.best_index_

# accuracy_migliore = random_search.cv_results_['mean_test_accuracy'][indice_migliore]
# loss_migliore = random_search.cv_results_['mean_test_neg_log_loss'][indice_migliore]

# print("\n--- PERFORMANCE DEL MODELLO VINCITORE (In Cross-Validation) ---")
# print(f"Migliore Accuracy Media: {accuracy_migliore:.4f}")
# print(f"Migliore Neg Log Loss Media: {loss_migliore:.4f} (Valore assoluto reale: {abs(loss_migliore):.4f})")

#### Predicting semifinals and finals

In [387]:
features.columns

Index(['year', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y', 'home_wc_titles_before',
       'home_total_market_value_eur', 'home_avg_age',
       'home_wc_participations_before', 'home_groups_passed_before',
       'home_round16_before', 'home_quarterfinals_before',
       'home_semifinals_before', 'home_finals_before', 'away_is_host',
       'away_goals_scored_last_4y', 'away_goals_received_last_4y',
       'away_wins_last_4y', 'away_losses_last_4y', 'away_draws_last_4y',
       'away_wc_titles_before', 'away_total_market_value_eur', 'away_avg_age',
       'away_wc_participations_before', 'away_groups_passed_before',
       'away_round16_before', 'away_quarterfinals_before',
       'away_semifinals_before', 'away_finals_before', 'stage', 'elo_home',
       'elo_away'],
      dtype='str')

In [388]:
# Hardcoding semifinals data
semifinals = pd.DataFrame({'home_is_host': [0,0], 'home_goals_scored_last_4y': [85,82],
       'home_goals_received_last_4y': [32,23], 'home_wins_last_4y': [25,26],
       'home_losses_last_4y': [6,6], 'home_draws_last_4y': [7,7], 'home_wc_titles_before': [2,1],
       'home_total_market_value_eur': [1290000000.0, 1300000000.0], 'home_avg_age': [26.7,26.8],
       'home_wc_participations_before': [16,16], 'home_groups_passed_before': [10,13],
       'home_round16_before': [8,8], 'home_quarterfinals_before': [9,10],
       'home_semifinals_before': [7,3], 'home_finals_before': [4,1], 'away_is_host': [0,0],
       'away_goals_scored_last_4y': [104,80], 'away_goals_received_last_4y': [32,14],
       'away_wins_last_4y': [29,30], 'away_losses_last_4y': [2,4], 'away_draws_last_4y': [8,3],
       'away_wc_titles_before': [1,3], 'away_total_market_value_eur': [1150000000.0,575000000.0], 'away_avg_age': [27.2,27.9],
       'away_wc_participations_before': [16,18], 'away_groups_passed_before': [11,15],
       'away_round16_before': [9,10], 'away_quarterfinals_before': [6,10],
       'away_semifinals_before': [2,6], 'away_finals_before': [1,6], 'stage' : [1,1], 'elo_home' : [1496.0, 1530.0], 'elo_away' : [1492.24, 1496.35]})
semifinals

,home_is_host,home_goals_scored_last_4y,home_goals_received_last_4y,home_wins_last_4y,home_losses_last_4y,home_draws_last_4y,home_wc_titles_before,home_total_market_value_eur,home_avg_age,home_wc_participations_before,...,away_avg_age,away_wc_participations_before,away_groups_passed_before,away_round16_before,away_quarterfinals_before,away_semifinals_before,away_finals_before,stage,elo_home,elo_away
0,0,85,32,25,6,7,2,1.290000e+09,26.7,16,...,27.2,16,11,9,6,2,1,1,1496.0,1492.24
1,0,82,23,26,6,7,1,1.300000e+09,26.8,16,...,27.9,18,15,10,10,6,6,1,1530.0,1496.35


In [389]:
# Predicting semifinals using XGBoost
prob = model.predict_proba(semifinals)

# creating a dataframe to store probabilities
df_prob = pd.DataFrame(prob, columns = ["home_prob", "away_prob", "draw_prob"])

# compute normalization for knockouts
sum_no_draw = df_prob["home_prob"] + df_prob["away_prob"]
df_prob["home_ko_prob"] = df_prob["home_prob"] / sum_no_draw
df_prob["away_ko_prob"] = df_prob["away_prob"] / sum_no_draw

def final_output(row):
    return [row["home_ko_prob"], row["away_ko_prob"]]

df_prob["probabilities"] = df_prob.apply(final_output, axis=1)

# inferring prediction from probabilities
def predict(row):
    if row["home_ko_prob"] > row["away_ko_prob"]:
        return 0 # home_team wins
    else:
        return 1 # away_team wins
    
df_prob["prediction"] = df_prob.apply(predict, axis=1)
preds = df_prob["prediction"]

In [390]:
preds = pd.DataFrame(preds) 
preds["home_team"] = ["France", "England"]
preds["away_team"] = ["Spain", "Argentina"]
preds

,prediction,home_team,away_team
0,0,France,Spain
1,1,England,Argentina


In [391]:
df_prob

,home_prob,away_prob,draw_prob,home_ko_prob,away_ko_prob,probabilities,prediction
0,0.529247,0.426324,0.044429,0.553854,0.446146,"[0.5538543, 0.4461457]",0
1,0.341217,0.568033,0.090750,0.375273,0.624727,"[0.37527323, 0.6247268]",1


In [392]:
# Hardcoding finals data
finals = pd.DataFrame({'home_is_host': [0,0], 'home_goals_scored_last_4y': [85,104],
       'home_goals_received_last_4y': [32,32], 'home_wins_last_4y': [25,29],
       'home_losses_last_4y': [6,2], 'home_draws_last_4y': [7,8], 'home_wc_titles_before': [2,1],
       'home_total_market_value_eur': [1290000000.0, 1150000000.0], 'home_avg_age': [26.7,27.2],
       'home_wc_participations_before': [16,16], 'home_groups_passed_before': [10,11],
       'home_round16_before': [8,9], 'home_quarterfinals_before': [9,6],
       'home_semifinals_before': [7,2], 'home_finals_before': [4,1], 'away_is_host': [0,0],
       'away_goals_scored_last_4y': [82,80], 'away_goals_received_last_4y': [23,14],
       'away_wins_last_4y': [26,30], 'away_losses_last_4y': [6,4], 'away_draws_last_4y': [7,3],
       'away_wc_titles_before': [1,3], 'away_total_market_value_eur': [1300000000.0,575000000.0], 'away_avg_age': [26.8,27.9],
       'away_wc_participations_before': [16,18], 'away_groups_passed_before': [13,15],
       'away_round16_before': [8,10], 'away_quarterfinals_before': [10,10],
       'away_semifinals_before': [3,6], 'away_finals_before': [1,6], 'stage' : [1,1], 'elo_home' : [1496.0, 1492.24], 'elo_away' : [1530.0, 1496.35]}
)
finals

,home_is_host,home_goals_scored_last_4y,home_goals_received_last_4y,home_wins_last_4y,home_losses_last_4y,home_draws_last_4y,home_wc_titles_before,home_total_market_value_eur,home_avg_age,home_wc_participations_before,...,away_avg_age,away_wc_participations_before,away_groups_passed_before,away_round16_before,away_quarterfinals_before,away_semifinals_before,away_finals_before,stage,elo_home,elo_away
0,0,85,32,25,6,7,2,1.290000e+09,26.7,16,...,26.8,16,13,8,10,3,1,1,1496.00,1530.00
1,0,104,32,29,2,8,1,1.150000e+09,27.2,16,...,27.9,18,15,10,10,6,6,1,1492.24,1496.35


In [393]:
# Predicting finals using XGBoost
prob = model.predict_proba(finals)

# creating a dataframe to store probabilities
df_prob = pd.DataFrame(prob, columns = ["home_prob", "away_prob", "draw_prob"])

# compute normalization for knockouts
sum_no_draw = df_prob["home_prob"] + df_prob["away_prob"]
df_prob["home_ko_prob"] = df_prob["home_prob"] / sum_no_draw
df_prob["away_ko_prob"] = df_prob["away_prob"] / sum_no_draw

def final_output(row):
    return [row["home_ko_prob"], row["away_ko_prob"]]

df_prob["probabilities"] = df_prob.apply(final_output, axis=1)

# inferring prediction from probabilities
def predict(row):
    if row["home_ko_prob"] > row["away_ko_prob"]:
        return 0 # home_team wins
    else:
        return 1 # away_team wins
    
df_prob["prediction"] = df_prob.apply(predict, axis=1)
preds = df_prob["prediction"]

In [394]:
preds = pd.DataFrame(preds) 
preds["home_team"] = ["France", "Spain"]
preds["away_team"] = ["England", "Argentina"]
preds

,prediction,home_team,away_team
0,0,France,England
1,0,Spain,Argentina


In [395]:
df_prob

,home_prob,away_prob,draw_prob,home_ko_prob,away_ko_prob,probabilities,prediction
0,0.534382,0.371558,0.094060,0.589865,0.410136,"[0.58986455, 0.4101355]",0
1,0.513242,0.392360,0.094399,0.566741,0.433259,"[0.5667413, 0.4332587]",0


In [396]:
# To be done: implementing a random forest